# GAM (pygam)

Modèle additif : prédiction = somme d'effets simples et lisibles (courbe sur l'heure de la semaine, courbe sur le mois, effet du type de jour, du férié, de l'événement, + une droite sur `lag_moy`). Un GAM par arc. pygam ne gère pas les NaN : on les remplace par la médiane du train.

In [ ]:
import sys; sys.path.append("..")  # pour importer dataset_create depuis notebooks/
import numpy as np
import pandas as pd
from dataset_create import charger, split, ORIGINES, FEATURES

df = charger(imputer=True)  # imputer=False : trous laissés en NaN
rmse = lambda y, p, axe: ((y - p) ** 2).groupby(axe, observed=True).mean() ** .5  # RMSE par arc

In [ ]:
from pygam import LinearGAM, s, f, l

COLS = ["heure_sem", "mois", "type_jour", "ferie", "evenement", "lag_moy"]
TERMES = s(0, n_splines=80) + s(1, n_splines=6) + f(2) + f(3) + f(4) + l(5)  # s = courbe, f = catégorie, l = droite

def modele(X_train, y_train, X_test):
    p = pd.Series(np.nan, X_test.index)
    for a in X_test.axe.unique():
        tr, te = X_train.axe == a, X_test.axe == a
        med = X_train.loc[tr, COLS].median()
        gam = LinearGAM(TERMES).fit(X_train.loc[tr, COLS].fillna(med), y_train[tr])
        p[te] = gam.predict(X_test.loc[te, COLS].fillna(med))
    return p.values

In [ ]:
# 1 semaine : dernière donnée le ven. 07/11/2025 -> on prédit sam. 08 -> mer. 12/11 (11/11 férié inclus)
CIBLE = "debit"  # ou "occupation"
X_train, X_test, y_train, y_test = split(df, "2025-11-07", cible=CIBLE)
p = modele(X_train, y_train, X_test)
print(pd.DataFrame({"modele": rmse(y_test, p, X_test.axe), "naif_lag7j": rmse(y_test, X_test.lag_7j, X_test.axe)}).round(1))

In [ ]:
# Backtest : même chose sur 1 vendredi sur 4 (ORIGINES pour tous), puis moyenne
res = []
for t0 in ORIGINES[::4]:
    X_train, X_test, y_train, y_test = split(df, t0, cible=CIBLE)
    if X_test.empty:  # aucun arc avec assez de vraies valeurs cette semaine-là
        continue
    p = modele(X_train, y_train, X_test)
    res.append(pd.DataFrame({"modele": rmse(y_test, p, X_test.axe), "naif_lag7j": rmse(y_test, X_test.lag_7j, X_test.axe)}))
par_arc = pd.concat(res).groupby(level=0).mean()
print(par_arc.round(1), "\n\nMoyenne des 3 arcs :\n", par_arc.mean().round(1))